In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import json
import os
import re
import subprocess
import sys

WORKSPACE = Path('/kaggle/working')
PROJECT_DIR = WORKSPACE / 'point_cloud_playground_play'
REPO_URL = 'https://github.com/kslannmnd/point_cloud_playground_play.git'
GITHUB_TOKEN = ''
GITHUB_TOKEN_SECRET = 'GITHUB_TOKEN'
S3DIS_PREPROCESSED_DIR = Path('/kaggle/input/datasets/ayukiseleva/s3dis-preprocessed')
TEST_AREA = 'Area_5'
N_TEST_ROOMS = 3
N_TRAIN_ROOMS = 6
PRETRAINED_METRIC_VARIANT = 'n_rooms'
TRAIN_VARIANT = 'n_rooms'
TRAINED_METRIC_VARIANT = 'n_rooms'
TRAIN_EXPERIMENT_NAME = 'kaggle_softgroup_s3dis'

def run(cmd, cwd=None, env=None, check=True, display_cmd=None):
    text = display_cmd or (' '.join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else str(cmd))
    print('\n[RUN]', text, '\n')
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, shell=isinstance(cmd, str), check=check)

def get_github_token():
    token = GITHUB_TOKEN.strip()
    if token:
        return token
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        return token.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET).strip()
    except Exception as exc:
        raise RuntimeError(f'Create Kaggle secret {GITHUB_TOKEN_SECRET!r} with private repo access first.') from exc

def authenticated_repo_url():
    token = get_github_token()
    if not token:
        raise RuntimeError('GitHub token is empty.')
    return REPO_URL.replace('https://', f'https://{token}@', 1)

assert S3DIS_PREPROCESSED_DIR.exists(), S3DIS_PREPROCESSED_DIR
for required_dir in ['preprocess', 'preprocess_sample', 'val_gt']:
    assert (S3DIS_PREPROCESSED_DIR / required_dir).exists(), S3DIS_PREPROCESSED_DIR / required_dir

## Clone and Install

In [ ]:
if PROJECT_DIR.exists():
    print('Project already exists:', PROJECT_DIR)
else:
    clone_url = authenticated_repo_url()
    run(['git', 'clone', clone_url, str(PROJECT_DIR)], display_cmd=f'git clone https://***@github.com/kslannmnd/point_cloud_playground_play.git {PROJECT_DIR}')

run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=PROJECT_DIR, check=False)

run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'wheel', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_DIR)

## Room Sets

In [ ]:
def natural_key(value):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', str(value))]

def discover_rooms(preprocess_dir):
    rooms = []
    for path in preprocess_dir.glob('*_inst_nostuff.pth'):
        match = re.match(r'^(Area_\d+)_(.+)_inst_nostuff\.pth$', path.name)
        if match:
            area, room = match.group(1), match.group(2)
            rooms.append({'area': area, 'room': room, 'scene': f'{area}_{room}'})
    return sorted(rooms, key=lambda item: (natural_key(item['area']), natural_key(item['room'])))

def room_ref(room):
    return f"{room['area']}/{room['room']}"

def list_override(name, rooms):
    return f"{name}=[{','.join(room_ref(room) for room in rooms)}]"

def metric_room_args(variant):
    if variant == 'all_test_rooms':
        return ['metrics.rooms=all_test_rooms']
    if variant == 'n_rooms':
        return [list_override('metrics.rooms', all_test_rooms[:N_TEST_ROOMS])]
    raise ValueError(variant)

def train_room_args(variant):
    if variant == 'all_trainable_rooms':
        return ['training.rooms=all_trainable_rooms']
    if variant == 'n_rooms':
        return [list_override('training.rooms', all_trainable_rooms[:N_TRAIN_ROOMS])]
    raise ValueError(variant)

all_rooms = discover_rooms(S3DIS_PREPROCESSED_DIR / 'preprocess')
all_test_rooms = [room for room in all_rooms if room['area'] == TEST_AREA]
all_trainable_rooms = [room for room in all_rooms if room['area'] != TEST_AREA]
assert all_test_rooms, TEST_AREA
assert all_trainable_rooms, TEST_AREA
print('all_test_rooms:', len(all_test_rooms))
print('all_trainable_rooms:', len(all_trainable_rooms))
print('metric n rooms:', [room['scene'] for room in all_test_rooms[:N_TEST_ROOMS]])
print('train n rooms:', [room['scene'] for room in all_trainable_rooms[:N_TRAIN_ROOMS]])

## Environment and Preprocessed S3DIS Link

In [ ]:
COMMON = [
    'env=kaggle',
    'dataset=s3dis_preprocessed',
    'inference=s3dis',
    f'env.kaggle.input_s3dis_preprocessed_dir={S3DIS_PREPROCESSED_DIR}',
    f'dataset.fold.preferred_test_area={TEST_AREA}',
]

run([sys.executable, 'scripts/prepare_kaggle_env.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/install_dependencies_and_build_softgroup.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/prepare_s3dis.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/download_softgroup_checkpoints.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/patch_softgroup_for_kaggle.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/generate_softgroup_configs.py', *COMMON], cwd=PROJECT_DIR)

## 1. Pretrained Metrics on Test Rooms

In [ ]:
PRETRAINED_CHECKPOINT = 'outputs/checkpoints/softgroup_s3dis_pretrained.pth'
pretrained_metric_cmd = [
    sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
    f'metrics.checkpoint={PRETRAINED_CHECKPOINT}',
    'metrics.force_run=true',
    'metrics.allow_precomputed_results=false',
    *metric_room_args(PRETRAINED_METRIC_VARIANT),
]
run(pretrained_metric_cmd, cwd=PROJECT_DIR)

## 2. Train SoftGroup on Trainable Rooms

In [ ]:
train_cmd = [
    sys.executable, 'scripts/train.py', *COMMON,
    'training=exp_1',
    f'training.experiment_name={TRAIN_EXPERIMENT_NAME}',
    'training.epochs.backbone=2',
    'training.epochs.full_softgroup=2',
    'training.lr.backbone=0.001',
    'training.lr.full_softgroup=0.001',
    'training.train_batch_size=1',
    'training.test_batch_size=1',
    'training.num_workers=2',
    *train_room_args(TRAIN_VARIANT),
]
run(train_cmd, cwd=PROJECT_DIR)
TRAINED_CHECKPOINT = f'outputs/checkpoints/{TRAIN_EXPERIMENT_NAME}_full_softgroup_latest.pth'
assert (PROJECT_DIR / TRAINED_CHECKPOINT).exists(), PROJECT_DIR / TRAINED_CHECKPOINT
print('Trained checkpoint:', PROJECT_DIR / TRAINED_CHECKPOINT)

## 3. Trained Metrics on Test Rooms

In [ ]:
trained_metric_cmd = [
    sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
    f'metrics.checkpoint={TRAINED_CHECKPOINT}',
    'metrics.force_run=true',
    'metrics.allow_precomputed_results=false',
    *metric_room_args(TRAINED_METRIC_VARIANT),
]
run(trained_metric_cmd, cwd=PROJECT_DIR)

## 4. S3DIS Test Room Inference with BBox HTML

In [ ]:
S3DIS_ROOM = all_test_rooms[0]
run([
    sys.executable, 'scripts/infer.py', *COMMON,
    f'inference.checkpoint={TRAINED_CHECKPOINT}',
    'inference.target.kind=room',
    f"inference.target.area={S3DIS_ROOM['area']}",
    f"inference.target.room={S3DIS_ROOM['room']}",
    'visualization.force_run=true',
    'visualization.allow_precomputed_results=false',
], cwd=PROJECT_DIR)
s3dis_runs = sorted([path for path in (PROJECT_DIR / 'outputs' / 'inference').iterdir() if path.is_dir()])
s3dis_html_files = sorted((s3dis_runs[-1] / 'html').glob('*.html'))
assert s3dis_html_files, s3dis_runs[-1]
s3dis_html_path = s3dis_html_files[-1]
print('S3DIS room:', S3DIS_ROOM['scene'])
print('S3DIS HTML:', s3dis_html_path)
display(HTML(s3dis_html_path.read_text(encoding='utf-8')))

## 5. R3D Frame Inference with Red-Point Boxes

In [ ]:
R3D_FILE = PROJECT_DIR / 'data' / 'r3d' / '2026-05-06--12-50-38.r3d'
assert R3D_FILE.exists(), R3D_FILE
run([
    sys.executable, 'run_r3d_inference.py', str(R3D_FILE),
    'env=kaggle',
    'dataset=r3d',
    'inference=r3d',
    f'inference.checkpoint={TRAINED_CHECKPOINT}',
], cwd=PROJECT_DIR)
r3d_runs = sorted([path for path in (PROJECT_DIR / 'outputs' / 'r3d_inference').iterdir() if path.is_dir()])
r3d_html_path = r3d_runs[-1] / 'point_cloud.html'
print('R3D HTML:', r3d_html_path)
display(HTML(r3d_html_path.read_text(encoding='utf-8')))

## Saved Artifacts

In [ ]:
metrics_root = PROJECT_DIR / 'outputs' / 'metrics' / 's3dis'
metric_runs = sorted([path for path in metrics_root.iterdir() if path.is_dir()]) if metrics_root.exists() else []
print('Metric runs:', len(metric_runs))
for path in metric_runs[-3:]:
    summary_path = path / 'summary.json'
    print(path)
    if summary_path.exists():
        print(json.loads(summary_path.read_text(encoding='utf-8')))
print('Checkpoint:', PROJECT_DIR / TRAINED_CHECKPOINT)
print('S3DIS HTML:', s3dis_html_path)
print('R3D HTML:', r3d_html_path)